In [ ]:
from helptools import register_user, login_user, identify
from PersonsDataset import PersonDataset
from dotenv import load_dotenv
from faker import Faker
import itertools as it
import random
import tqdm

In [ ]:
load_dotenv()

In [ ]:
train_dataset = PersonDataset("../data/vggface2/train")

In [ ]:
fake = Faker()
subset = list(it.islice(train_dataset, 200))
train_check = {}
for images, person_id in tqdm.tqdm(subset):
    first_name = fake.unique.first_name()
    last_name = fake.unique.last_name()

    response = await register_user(
        images=images[3:], first_name=first_name, last_name=last_name
    )

    id = response.json().get("id")
    if id is not None:
        train_check[id] = images[:2]

In [ ]:
FAR = 0  # False acceptance rate
FRR = 0  # False reject rate
TAR = 0  # True acceptance rate
TRR = 0  # True reject rate

In [ ]:
ids = train_check.keys()

for id, images in train_check.items():
    for img in images:
        r = await login_user(img, id)
        s = r.json().get('success')
        if s is not None:
            if s:
                TAR += 1
            else:
                FRR += 1

        other_id = random.choice(ids)
        while other_id == id:
            other_id = random.choice(ids)

        r = await login_user(img, other_id)
        s = r.json().get('success')
        if s is not None:
            if s:
                FAR += 1
            else:
                TRR += 1

In [ ]:
val_dataset = PersonDataset("../data/vggface2/val")

In [ ]:
fake = Faker()
subset = list(it.islice(train_dataset, 10))
possible_attacks = 0
for images, person_id in tqdm.tqdm(subset):

    responses = []

    for image in images:
        responses += [await identify(image) for image in images]

    contents = list(map(lambda resp: resp.json().get("found", []), responses))

    possible_attacks += len(set(sum(contents, [])))